# 02 - CSV para Delta Lake (MinIO)

Lê os arquivos CSV do bucket `landing-zone` no MinIO e grava cada um como tabela Delta Lake no bucket `bronze`.

**Pré-requisitos:** Notebook `01` executado (CSVs no MinIO).

## 1. Imports e Configuração

In [2]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

print(f'MinIO: {MINIO_ENDPOINT}')
print(f'Landing: {LANDING_BUCKET} | Bronze: {BRONZE_BUCKET}')

MinIO: http://localhost:9020
Landing: landing-zone | Bronze: bronze


## 2. Criar SparkSession com Delta Lake e MinIO

In [3]:
spark = (
    SparkSession.builder
    .appName('CSV_to_Delta')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # MinIO / S3A
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

your 131072x1 screen size is bogus. expect trouble
26/04/28 12:30:16 WARN Utils: Your hostname, NOTEDELL3420 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/28 12:30:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/jlsilva01/codigos/spark-delta-minio-sqlserver/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jlsilva01/.ivy2/cache
The jars for the packages stored in: /home/jlsilva01/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-64dbb3b8-20a8-47ed-b23e-6fe0eb0b1436;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 497ms :: artifacts dl 18ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apach

SparkSession criada com sucesso!


## 3. Criar Bucket Bronze no MinIO

In [4]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] ja existe')
except:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [bronze] criado!
Buckets: ['bronze', 'landing-zone']


## 4. Listar CSVs Disponíveis no Landing Zone

In [5]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
csv_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

print(f'{len(csv_files)} arquivos CSV encontrados no bucket [{LANDING_BUCKET}]:')
for f in csv_files:
    print(f'  - {f}')

11 arquivos CSV encontrados no bucket [landing-zone]:
  - apolice.csv
  - carro.csv
  - cliente.csv
  - endereco.csv
  - estado.csv
  - marca.csv
  - modelo.csv
  - municipio.csv
  - regiao.csv
  - sinistro.csv
  - telefone.csv


## 5. Ler CSVs e Gravar como Delta Lake

In [6]:
from delta.tables import DeltaTable

print(f'Convertendo {len(csv_files)} CSVs para Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    csv_path = f's3a://{LANDING_BUCKET}/{csv_file}'
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # Ler CSV com inferência de schema
    df = spark.read \
        .option('header', 'true') \
        .option('inferSchema', 'true') \
        .csv(csv_path)
    
    # Gravar como Delta Lake
    df.write \
        .format('delta') \
        .mode('overwrite') \
        .save(delta_path)
    
    print(f'  {tabela}: {df.count()} registros | {len(df.columns)} colunas -> {delta_path}')

print(f'\nConversao concluida! {len(csv_files)} tabelas Delta criadas no bucket [{BRONZE_BUCKET}].')

Convertendo 11 CSVs para Delta Lake...



26/04/28 12:30:24 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


  apolice: 10000 registros | 7 colunas -> s3a://bronze/apolice


  carro: 10002 registros | 5 colunas -> s3a://bronze/carro


  cliente: 20010 registros | 5 colunas -> s3a://bronze/cliente
  endereco: 20010 registros | 5 colunas -> s3a://bronze/endereco
  estado: 27 registros | 4 colunas -> s3a://bronze/estado
  marca: 10 registros | 2 colunas -> s3a://bronze/marca
  modelo: 100 registros | 3 colunas -> s3a://bronze/modelo
  municipio: 5570 registros | 3 colunas -> s3a://bronze/municipio
  regiao: 5 registros | 2 colunas -> s3a://bronze/regiao
  sinistro: 10000 registros | 5 colunas -> s3a://bronze/sinistro
  telefone: 20010 registros | 2 colunas -> s3a://bronze/telefone

Conversao concluida! 11 tabelas Delta criadas no bucket [bronze].


## 6. Validação - Ler Tabelas Delta

In [7]:
print('Validando tabelas Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # Verificar se é Delta
    is_delta = DeltaTable.isDeltaTable(spark, delta_path)
    df_delta = spark.read.format('delta').load(delta_path)
    
    print(f'  {tabela}: Delta={is_delta} | {df_delta.count()} registros | Colunas: {df_delta.columns}')

Validando tabelas Delta Lake...



26/04/28 12:30:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  apolice: Delta=True | 10000 registros | Colunas: ['cd_apolice', 'cd_cliente', 'dt_inicio_vigencia', 'dt_fim_vigencia', 'vl_cobertura', 'vl_franquia', 'placa']


  carro: Delta=True | 10002 registros | Colunas: ['placa', 'cd_modelo', 'chassi', 'ano', 'cor']


  cliente: Delta=True | 20010 registros | Colunas: ['cd_cliente', 'nome', 'cpf', 'sexo', 'dt_nascimento']


...

  modelo: Delta=True | 100 registros | Colunas: ['cd_modelo', 'cd_marca', 'nm_modelo']


  municipio: Delta=True | 5570 registros | Colunas: ['cd_municipio', 'nm_municipio', 'cd_estado']


  regiao: Delta=True | 5 registros | Colunas: ['cd_regiao', 'nm_regiao']


  sinistro: Delta=True | 10000 registros | Colunas: ['cd_sinistro', 'placa', 'dt_sinistro', 'local_sinistro', 'condutor']


  telefone: Delta=True | 20010 registros | Colunas: ['cd_cliente', 'nr_telefone']


In [8]:
# Amostra: exibir primeiros registros de algumas tabelas
for tabela in ['regiao', 'marca', 'cliente']:
    print(f'\n--- {tabela.upper()} ---')
    spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/{tabela}').show(5)


--- REGIAO ---


+---------+------------+
|cd_regiao|   nm_regiao|
+---------+------------+
|        1|       NORTE|
|        2|    NORDESTE|
|        3|     SUDESTE|
|        4|         SUL|
|        5|CENTRO-OESTE|
+---------+------------+


--- MARCA ---


+--------+----------+
|cd_marca|  nm_marca|
+--------+----------+
|       1| CHEVROLET|
|       2|VOLKSWAGEN|
|       3|      FIAT|
|       4|      FORD|
|       5|    TOYOTA|
+--------+----------+
only showing top 5 rows


--- CLIENTE ---


+----------+--------------------+-----------+----+-------------+
|cd_cliente|                nome|        cpf|sexo|dt_nascimento|
+----------+--------------------+-----------+----+-------------+
|         1|MARISA MELO OLIVEIRA|54367845687|   F|   2000-09-06|
|         2|MURILO CARVALHO C...|34563456566|   M|   1989-06-26|
|         3|VINICIUS ROCHA RO...|83946466633|   M|   1990-01-25|
|         4|CAROLINA ROCHA GOMES| 9837746366|   F|   1996-04-21|
|         5| ALINE SANTOS CASTRO|99934377292|   F|   1962-07-04|
+----------+--------------------+-----------+----+-------------+
only showing top 5 rows



In [9]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
